In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql("USE CATALOG fvp_lab")

In [0]:
requested_policy_id = "P003"

In [0]:
policy_consent = (
    spark.table(
        "fvp_lab.streaming.policy_consent"
    )
    .filter(
        F.col("policyId") == requested_policy_id
    )
)


In [0]:
consent = policy_consent.collect()

In [0]:
if len(consent) == 0:

    print(
        "Documento não autorizado "
        "(sem consentimento)"
    )

else:

    consent_row = consent[0]

    if consent_row["allowed"] is False:

        print(
            "Documento bloqueado"
        )

    else:

        consent_id = (
            consent_row["consentId"]
        )

        # ==========================
        # CONSULTA COSMOS
        # ==========================

        cosmos_df = (
            spark.table(
                "fvp_lab.streaming.cosmos_policy"
            )
            .filter(
                F.col("policyId")
                == requested_policy_id
            )
        )

        # ==========================
        # SUBSTITUI NULL
        # ==========================

        response_df = (
            cosmos_df
            .withColumn(
                "consentId",
                F.lit(consent_id)
            )
        )

        print(
            "Documento retornado:"
        )

        display(response_df)